In [ ]:
# 1. Import Libraries
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pathlib
from tensorflow.keras import layers, models
print("TensorFlow Version:", tf.__version__)

In [ ]:
# 2. Download and Load Flower Dataset
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
path = tf.keras.utils.get_file(
    "flower_photos.tgz",
    url,
    extract=True
)
# Find correct dataset directory
search_dir = pathlib.Path(path).parent
folders = list(search_dir.rglob("daisy"))
if not folders:
    raise FileNotFoundError("Flower dataset not found. Check internet connection.")
data_dir = folders[0].parent
# Load training dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(160, 160),
    batch_size=32
)
# Load validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(160, 160),
    batch_size=32
)
class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
# 3. Display Sample Images
plt.figure(figsize=(8, 5))
for images, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(2, 3, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Load Pre-trained MobileNetV2
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet"
)
# Freeze pre-trained layers
base_model.trainable = False

In [ ]:
# 5. Build CNN Model
model = models.Sequential([
    layers.Input(shape=(160, 160, 3)),
    layers.RandomFlip("horizontal"),
    layers.Rescaling(1./127.5, offset=-1),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(5, activation="softmax")
])

In [ ]:
# 6. Compile Model
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

In [ ]:
# 7. Train Model (Transfer Learning)
print("Training the model...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

In [ ]:
# 8. Fine-Tuning
print("Fine-tuning the model...")
base_model.trainable = True
# Unfreeze only last 20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False
# Keep BatchNormalization layers frozen
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)

In [ ]:
# 9. Evaluate Model
loss, accuracy = model.evaluate(val_ds)
print("Validation Accuracy: {:.2f}%".format(accuracy * 100))

In [ ]:
# 10. Display Image Predictions
plt.figure(figsize=(12, 7))
for images, labels in val_ds.take(1):
    predictions = model.predict(images, verbose=0)
    for i in range(8):
        plt.subplot(2, 4, i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        predicted = np.argmax(predictions[i])
        actual = int(labels[i])
        confidence = predictions[i][predicted] * 100
        plt.title(
            f"Predicted: {class_names[predicted]}\n"
            f"Actual: {class_names[actual]}\n"
            f"Confidence: {confidence:.1f}%",
            color="green" if predicted == actual else "red",
            fontsize=9
        )
        plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 11. Save Model
model.save("flower_classifier.keras")
print("Model saved successfully!")